### Layout Benchmark Evaluation — DocLayNet

Đánh giá 3 model Layout (**`docling-layout-heron`**, **`DocLayout-YOLO`**, **`PP-DocLayoutV3`**) trên **4.999 trang DocLayNet** (9 lớp nhãn chung).

* **`F1@0.50` / `F1@0.75`:** Trung bình điều hòa giữa *Precision* và *Recall* tại ngưỡng `IoU ≥ 0.50` / `0.75`.
* **`Precision@0.50` / `Recall@0.50`:** Tỷ lệ khung dự đoán đúng (*Precision*) và tỷ lệ khung đáp án được phát hiện (*Recall*) tại `IoU ≥ 0.50`.
* **`mAP@0.50:0.95` / `AP@0.50` / `AP@0.75`:** Độ chính xác trung bình trên tất cả các lớp qua 10 ngưỡng `IoU` từ `0.50` đến `0.95`, và riêng tại `0.50` / `0.75`.
* **`IoU` *(Intersection over Union)*:** Tỷ lệ diện tích phần giao trên phần hợp giữa khung dự đoán và khung đáp án.

In [1]:
import json
from collections import defaultdict
from pathlib import Path
from time import perf_counter

import numpy as np
import pandas as pd
from IPython.display import display

IOU_THRESHOLDS = np.linspace(0.50, 0.95, 10)
REC_THRESHOLDS = np.linspace(0.0, 1.0, 101)
MAX_DETS = 100


def compute_iou_matrix(boxes_a: np.ndarray, boxes_b: np.ndarray) -> np.ndarray:
    if len(boxes_a) == 0 or len(boxes_b) == 0:
        return np.zeros((len(boxes_a), len(boxes_b)), dtype=np.float64)
    x1 = np.maximum(boxes_a[:, None, 0], boxes_b[None, :, 0])
    y1 = np.maximum(boxes_a[:, None, 1], boxes_b[None, :, 1])
    x2 = np.minimum(boxes_a[:, None, 2], boxes_b[None, :, 2])
    y2 = np.minimum(boxes_a[:, None, 3], boxes_b[None, :, 3])
    inter = np.maximum(0.0, x2 - x1) * np.maximum(0.0, y2 - y1)
    area_a = np.maximum(0.0, boxes_a[:, 2] - boxes_a[:, 0]) * np.maximum(0.0, boxes_a[:, 3] - boxes_a[:, 1])
    area_b = np.maximum(0.0, boxes_b[:, 2] - boxes_b[:, 0]) * np.maximum(0.0, boxes_b[:, 3] - boxes_b[:, 1])
    union = area_a[:, None] + area_b[None, :] - inter
    return np.where(union > 0.0, inter / union, 0.0)


def extract_docling_boxes(doc: dict, gt_width: float, gt_height: float, label_map: dict[str, str]) -> dict[str, list[list[float]]]:
    pages = doc.get("pages") or {}
    page_info = pages.get("1") or (next(iter(pages.values())) if pages else None)
    if not page_info or "size" not in page_info:
        return {}
    page_w = float(page_info["size"]["width"])
    page_h = float(page_info["size"]["height"])
    if page_w <= 0 or page_h <= 0:
        return {}
    scale_x, scale_y = gt_width / page_w, gt_height / page_h

    by_cat: dict[str, list[list[float]]] = defaultdict(list)
    for section in ("texts", "tables", "pictures", "key_value_items"):
        for item in doc.get(section) or ():
            raw_label = str(item.get("label", "")).strip().lower()
            mapped = label_map.get(raw_label)
            if mapped is None:
                continue
            provs = item.get("prov") or ()
            if not provs:
                continue
            bbox = provs[0].get("bbox")
            if not bbox:
                continue
            l, t, r, b = float(bbox["l"]), float(bbox["t"]), float(bbox["r"]), float(bbox["b"])
            origin = str(bbox.get("coord_origin", "BOTTOMLEFT")).upper()
            y1, y2 = (page_h - t, page_h - b) if origin == "BOTTOMLEFT" else (t, b)
            x1, x2 = min(l, r) * scale_x, max(l, r) * scale_x
            y1, y2 = min(y1, y2) * scale_y, max(y1, y2) * scale_y
            x1, x2 = min(max(x1, 0.0), gt_width), min(max(x2, 0.0), gt_width)
            y1, y2 = min(max(y1, 0.0), gt_height), min(max(y2, 0.0), gt_height)
            if x2 > x1 and y2 > y1:
                by_cat[mapped].append([x1, y1, x2, y2])
    return by_cat


def compute_ap_101(tp_flags: np.ndarray, total_gt: int) -> float:
    if total_gt == 0:
        return float("nan")
    if len(tp_flags) == 0:
        return 0.0
    tp_cum = np.cumsum(tp_flags, dtype=np.float64)
    fp_cum = np.cumsum(1.0 - tp_flags, dtype=np.float64)
    rec = tp_cum / total_gt
    prec = tp_cum / np.maximum(tp_cum + fp_cum, 1e-12)
    for i in range(len(prec) - 1, 0, -1):
        if prec[i] > prec[i - 1]:
            prec[i - 1] = prec[i]
    inds = np.searchsorted(rec, REC_THRESHOLDS, side="left")
    q = np.zeros(len(REC_THRESHOLDS), dtype=np.float64)
    valid = inds < len(prec)
    q[valid] = prec[inds[valid]]
    return float(np.mean(q))


def prf1(tp: int, fp: int, fn: int) -> tuple[float, float, float]:
    p = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    r = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2 * p * r / (p + r) if (p + r) > 0 else 0.0
    return p, r, f1


def evaluate_subset(image_ids, gt_by_img_cat, pred_by_img_cat, cat_names):
    n_t = len(IOU_THRESHOLDS)
    idx_50, idx_75 = 0, 5
    per_cat_ap, per_cat_f1_50 = {}, {}
    total_tp_50 = total_fp_50 = total_fn_50 = 0
    total_tp_75 = total_fp_75 = total_fn_75 = 0

    for cname in cat_names:
        tp_lists = [[] for _ in range(n_t)]
        total_gt = 0
        cat_tp_50 = cat_fp_50 = cat_tp_75 = cat_fp_75 = 0

        for img_id in image_ids:
            gt_boxes = gt_by_img_cat.get((img_id, cname))
            pred_boxes = pred_by_img_cat.get((img_id, cname))
            n_gt = 0 if gt_boxes is None else len(gt_boxes)
            n_pred = 0 if pred_boxes is None else min(len(pred_boxes), MAX_DETS)
            total_gt += n_gt
            if n_pred == 0:
                continue
            if n_gt == 0:
                for t_idx in range(n_t):
                    tp_lists[t_idx].extend([0.0] * n_pred)
                cat_fp_50 += n_pred
                cat_fp_75 += n_pred
                continue

            ious = compute_iou_matrix(pred_boxes[:n_pred], gt_boxes)
            for t_idx, thr in enumerate(IOU_THRESHOLDS):
                matched_gt = np.zeros(n_gt, dtype=bool)
                tp_img = 0
                for d_idx in range(n_pred):
                    row = ious[d_idx]
                    best_g, best_iou = -1, thr - 1e-12
                    for g_idx in range(n_gt):
                        if not matched_gt[g_idx] and row[g_idx] >= best_iou:
                            best_iou = row[g_idx]
                            best_g = g_idx
                    if best_g >= 0:
                        matched_gt[best_g] = True
                        tp_lists[t_idx].append(1.0)
                        tp_img += 1
                    else:
                        tp_lists[t_idx].append(0.0)
                if t_idx == idx_50:
                    cat_tp_50 += tp_img
                    cat_fp_50 += n_pred - tp_img
                elif t_idx == idx_75:
                    cat_tp_75 += tp_img
                    cat_fp_75 += n_pred - tp_img

        aps = [compute_ap_101(np.asarray(tp_lists[t_idx], dtype=np.float64), total_gt) for t_idx in range(n_t)]
        per_cat_ap[cname] = {
            "mAP_50_95": float(np.nanmean(aps)) if not all(np.isnan(aps)) else float("nan"),
            "AP_50": aps[idx_50],
            "AP_75": aps[idx_75],
            "gt_count": total_gt,
        }
        cat_fn_50 = total_gt - cat_tp_50
        cat_fn_75 = total_gt - cat_tp_75
        _, _, per_cat_f1_50[cname] = prf1(cat_tp_50, cat_fp_50, cat_fn_50)
        total_tp_50 += cat_tp_50
        total_fp_50 += cat_fp_50
        total_fn_50 += cat_fn_50
        total_tp_75 += cat_tp_75
        total_fp_75 += cat_fp_75
        total_fn_75 += cat_fn_75

    valid_map = [v["mAP_50_95"] for v in per_cat_ap.values() if not np.isnan(v["mAP_50_95"])]
    valid_ap50 = [v["AP_50"] for v in per_cat_ap.values() if not np.isnan(v["AP_50"])]
    valid_ap75 = [v["AP_75"] for v in per_cat_ap.values() if not np.isnan(v["AP_75"])]
    p50, r50, f1_50 = prf1(total_tp_50, total_fp_50, total_fn_50)
    _, _, f1_75 = prf1(total_tp_75, total_fp_75, total_fn_75)
    return {
        "mAP_50_95": float(np.mean(valid_map)) if valid_map else 0.0,
        "AP_50": float(np.mean(valid_ap50)) if valid_ap50 else 0.0,
        "AP_75": float(np.mean(valid_ap75)) if valid_ap75 else 0.0,
        "P_50": p50,
        "R_50": r50,
        "F1_50": f1_50,
        "F1_75": f1_75,
        "per_cat_ap": per_cat_ap,
        "per_cat_f1_50": per_cat_f1_50,
    }


def fmt(val: float) -> str:
    return "-" if np.isnan(val) else f"{val * 100:.2f}%"

BASE_DIR = Path(".")
GT_PATH = BASE_DIR / "ground_truth.json"
MODELS = ["docling-layout-heron", "DocLayout-YOLO", "PP-DocLayoutV3"]

# Chuẩn 9 nhãn chung đồng bộ với OmniDocBench EN (gộp Title -> Section-header và List-item -> Text)
GT_LABEL_MERGE = {
    "Title": "Section-header",
    "List-item": "Text",
}

DOCLING_LABEL_MAP = {
    "caption": "Caption",
    "footnote": "Footnote",
    "formula": "Formula",
    "list_item": "Text",
    "page_footer": "Page-footer",
    "page_header": "Page-header",
    "picture": "Picture",
    "chart": "Picture",
    "section_header": "Section-header",
    "title": "Section-header",
    "table": "Table",
    "document_index": "Table",
    "text": "Text",
    "paragraph": "Text",
    "code": "Text",
    "reference": "Text",
}

t0 = perf_counter()
gt_data = json.loads(GT_PATH.read_text(encoding="utf-8"))
id_to_name = {
    c["id"]: GT_LABEL_MERGE.get(c["name"], c["name"])
    for c in gt_data["categories"]
}
CAT_NAMES = sorted(set(id_to_name.values()))

images_by_id = {img["id"]: img for img in gt_data["images"]}
doc_cat_images = defaultdict(list)
for img in gt_data["images"]:
    doc_cat_images[img.get("doc_category", "unknown")].append(img["id"])
doc_categories = sorted(doc_cat_images)

gt_lists = defaultdict(list)
for ann in gt_data["annotations"]:
    if ann.get("iscrowd", 0):
        continue
    x, y, w, h = [float(v) for v in ann["bbox"]]
    if w > 0 and h > 0:
        gt_lists[(ann["image_id"], id_to_name[ann["category_id"]])].append([x, y, x + w, y + h])
gt_by_img_cat = {k: np.asarray(v, dtype=np.float64) for k, v in gt_lists.items()}
all_image_ids = sorted(images_by_id)

results, doc_cat_results = {}, {}
for m in MODELS:
    mdir = BASE_DIR / m
    if not mdir.exists():
        print(f"[SKIP] Không tìm thấy: {mdir}")
        continue
    pred_by_img_cat = {}
    missing = 0
    for img_id, img in images_by_id.items():
        pred_path = mdir / Path(img["file_name"]).with_suffix(".json")
        if not pred_path.exists():
            missing += 1
            continue
        doc = json.loads(pred_path.read_text(encoding="utf-8"))
        for cname, boxes in extract_docling_boxes(doc, float(img["width"]), float(img["height"]), DOCLING_LABEL_MAP).items():
            pred_by_img_cat[(img_id, cname)] = np.asarray(boxes, dtype=np.float64)
    results[m] = evaluate_subset(all_image_ids, gt_by_img_cat, pred_by_img_cat, CAT_NAMES)
    doc_cat_results[m] = {
        dcat: evaluate_subset(doc_cat_images[dcat], gt_by_img_cat, pred_by_img_cat, CAT_NAMES)
        for dcat in doc_categories
    }
    print(f"✓ {m}: missing={missing} | mAP@50:95={fmt(results[m]['mAP_50_95'])} | F1@50={fmt(results[m]['F1_50'])}")

print(f"Hoàn tất đánh giá {len(all_image_ids)} trang trong {perf_counter() - t0:.1f}s")

✓ docling-layout-heron: missing=0 | mAP@50:95=55.75% | F1@50=88.25%
✓ DocLayout-YOLO: missing=0 | mAP@50:95=26.13% | F1@50=69.11%
✓ PP-DocLayoutV3: missing=0 | mAP@50:95=28.37% | F1@50=77.88%
Hoàn tất đánh giá 4999 trang trong 25.6s


In [2]:
# Bảng 1: Tổng quan (Overall)
rows_overall = []
for m in MODELS:
    if m not in results:
        continue
    r = results[m]
    rows_overall.append({
        "Model": m,
        "mAP@0.50:0.95": fmt(r["mAP_50_95"]),
        "AP@0.50": fmt(r["AP_50"]),
        "AP@0.75": fmt(r["AP_75"]),
        "F1@0.50": fmt(r["F1_50"]),
        "Precision@0.50": fmt(r["P_50"]),
        "Recall@0.50": fmt(r["R_50"]),
        "F1@0.75": fmt(r["F1_75"]),
    })
df_overall = pd.DataFrame(rows_overall)
display(df_overall)

,Model,mAP@0.50:0.95,AP@0.50,AP@0.75,F1@0.50,Precision@0.50,Recall@0.50,F1@0.75
0,docling-layout-heron,55.75%,75.34%,60.62%,88.25%,86.83%,89.73%,81.25%
1,DocLayout-YOLO,26.13%,44.58%,22.34%,69.11%,73.61%,65.12%,49.88%
2,PP-DocLayoutV3,28.37%,47.50%,24.68%,77.88%,75.77%,80.10%,58.69%


In [3]:
# Bảng 2: Chi tiết theo từng loại Layout (F1@0.50 và COCO AP@0.50:0.95)
eval_models = [m for m in MODELS if m in results]

rows_f1 = []
rows_ap = []
for cname in CAT_NAMES:
    gt_cnt = results[eval_models[0]]["per_cat_ap"][cname]["gt_count"]
    row_f1 = {"Layout Class": cname, "GT Boxes": gt_cnt}
    row_ap = {"Layout Class": cname, "GT Boxes": gt_cnt}
    for m in eval_models:
        row_f1[m] = fmt(results[m]["per_cat_f1_50"][cname])
        row_ap[m] = fmt(results[m]["per_cat_ap"][cname]["mAP_50_95"])
    rows_f1.append(row_f1)
    rows_ap.append(row_ap)

print("=== Bảng 2a: Per-Class F1@0.50 ===")
display(pd.DataFrame(rows_f1))
print("=== Bảng 2b: Per-Class COCO AP@0.50:0.95 ===")
display(pd.DataFrame(rows_ap))

=== Bảng 2a: Per-Class F1@0.50 ===


,Layout Class,GT Boxes,docling-layout-heron,DocLayout-YOLO,PP-DocLayoutV3
0,Caption,1543,83.24%,55.89%,58.18%
1,Footnote,387,80.45%,21.78%,37.66%
2,Formula,1966,91.81%,78.45%,42.33%
3,Page-footer,3994,85.37%,61.27%,63.65%
4,Page-header,3366,88.47%,46.83%,61.48%
5,Picture,3534,80.77%,55.42%,59.34%
6,Section-header,8885,86.40%,80.35%,81.26%
7,Table,2394,86.98%,85.08%,85.44%
8,Text,40462,89.77%,70.16%,85.56%


=== Bảng 2b: Per-Class COCO AP@0.50:0.95 ===


,Layout Class,GT Boxes,docling-layout-heron,DocLayout-YOLO,PP-DocLayoutV3
0,Caption,1543,53.53%,13.31%,19.81%
1,Footnote,387,51.11%,2.79%,11.30%
2,Formula,1966,56.05%,33.00%,8.14%
3,Page-footer,3994,39.09%,10.41%,13.34%
4,Page-header,3366,53.95%,9.86%,17.65%
5,Picture,3534,58.35%,32.34%,32.50%
6,Section-header,8885,48.02%,29.46%,33.10%
7,Table,2394,72.27%,69.25%,66.65%
8,Text,40462,69.36%,34.79%,52.80%


In [4]:
# Bảng 3: Chi tiết theo từng loại PDF
eval_models = [m for m in MODELS if m in results]
rows_pdf = []
for dcat in doc_categories:
    row = {"PDF Type": dcat, "Pages": len(doc_cat_images[dcat])}
    for m in eval_models:
        r = doc_cat_results[m][dcat]
        row[f"{m} (mAP / F1@50)"] = f"{fmt(r['mAP_50_95'])} / {fmt(r['F1_50'])}"
    rows_pdf.append(row)

df_pdf = pd.DataFrame(rows_pdf)
display(df_pdf)

,PDF Type,Pages,docling-layout-heron (mAP / F1@50),DocLayout-YOLO (mAP / F1@50),PP-DocLayoutV3 (mAP / F1@50)
0,financial_reports,1755,44.82% / 87.10%,24.65% / 68.46%,28.10% / 75.97%
1,government_tenders,273,49.17% / 88.10%,31.26% / 67.55%,41.71% / 85.46%
2,laws_and_regulations,784,40.09% / 83.79%,18.73% / 66.76%,29.75% / 82.85%
3,manuals,802,54.07% / 89.07%,30.10% / 67.80%,36.34% / 84.63%
4,patents,442,59.90% / 83.79%,19.25% / 63.14%,19.55% / 61.68%
5,scientific_articles,943,76.84% / 96.81%,39.72% / 77.76%,38.48% / 79.90%
